In [1]:
REPO = '../'
RESULT_TABLE = f'{REPO}/data/table'
RESULT_OBJ = f'{REPO}/data/object'
FIGURE_FOLDER= f'{REPO}/data/figure'
SETTING_FOLDER = f'{REPO}/data/setting'
EXTERNAL_DATA=f'{REPO}/data/external'
# load Terra API for get data table from Terra
import sys
sys.path.append(REPO)
from settings import COLOR_PAlETTE
from utils.visual import *

import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scPipe as sp
import scanpy as sc
import seaborn as sns
import warnings
import commentjson
### Additional Colormap
with open(f'{SETTING_FOLDER}/ColorMap.json','r') as f:
    colormap = commentjson.load(f)
COLOR_PAlETTE.update(colormap)
plt.style.use(f'{REPO}/paper.mplstyle')
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
obs_path = f'{RESULT_TABLE}/annotation/Epithelial.csv'
adata = sc.read(f'{RESULT_OBJ}/gex_all.h5ad')
obs = pd.read_csv(obs_path,index_col=0)
adata = adata[adata.obs.index.isin(obs.index[obs.Lineage=='Tumor']),:]
print(f"Detect N={adata.shape[0]:,} tumor cells with high quality GEX profiles;")
print(f"max(% of mitocondrial reads): {adata.obs['pct_counts_mito'].max():.0f}")
sample_meta =  pd.read_excel(f'{RESULT_TABLE}/Supplementary Table 1.xlsx',index_col=0).replace(np.nan,'N/A')

Detect N=65,206 tumor cells with high quality GEX profiles;
max(% of mitocondrial reads): 5


# Random split tumor cells 1:1 per sample

In [3]:
split_meta = pd.Series()
for _,df in adata.obs.groupby('Sample'):
    n = df.shape[0] 
    half1 = df.sample(frac=.5,random_state=42)
    split_meta = split_meta.append(pd.Series(['Tumor_Split_1']*half1.shape[0],index=half1.index))
    half2 = df.drop(half1.index)
    split_meta = split_meta.append(pd.Series(['Tumor_Split_2']*half2.shape[0],index=half2.index))
split_meta.name = 'TumorRandomSplit'
split_meta.index.name = 'barcodes'
print(split_meta.value_counts())
print(split_meta.to_frame().head())
split_meta.to_frame().to_csv(f'{RESULT_TABLE}/TumorRandom.csv',index=True)

Tumor_Split_1    32607
Tumor_Split_2    32599
Name: TumorRandomSplit, dtype: int64
                                                   TumorRandomSplit
barcodes                                                           
TACAGCTAGGATGATG-1_CCG1112_01_T1_A_CCG1112_MO_B...    Tumor_Split_1
CGCTTCTAGGCCTTAG-1_CCG1112_01_T1_A_CCG1112_MO_B...    Tumor_Split_1
CTAAGGTTCATTGCGG-1_CCG1112_01_T1_A_CCG1112_MO_B...    Tumor_Split_1
GCCTGACAGCATGGTT-1_CCG1112_01_T1_A_CCG1112_MO_B...    Tumor_Split_1
TACGGTTAGCTAAAGG-1_CCG1112_01_T1_A_CCG1112_MO_B...    Tumor_Split_1


# 95% 10 times

In [6]:
splits = {}
for i in range(10):
    splits[f'TumorRandom95_{i}'] = pd.Series()
    for sample, df in adata.obs.groupby('Sample'):
        sampled_indices = df.sample(frac=.95).index
        splits[f'TumorRandom95_{i}'] = splits[f'TumorRandom95_{i}'].append(pd.Series(
            ['selected'] * len(sampled_indices),
            index=sampled_indices,
            name=f'TumorRandom95_{i}'
        ))
result = pd.DataFrame(splits).fillna('unselected')
result.index.name = 'barcodes'
result.to_csv(f'{RESULT_TABLE}/TumorRandom95_10times.csv',index=True)
result.head()

,TumorRandom95_0,TumorRandom95_1,TumorRandom95_2,TumorRandom95_3,TumorRandom95_4,TumorRandom95_5,TumorRandom95_6,TumorRandom95_7,TumorRandom95_8,TumorRandom95_9
barcodes,,,,,,,,,,
AAACAGCCAAATTGCT-1_CCG1112_01_T1_A_CCG1112_MO_Batch3,selected,selected,selected,selected,selected,selected,selected,selected,selected,selected
AAACAGCCAACACTTG-1_CCG1112_26_T1_B1_CCG1112_MO_Batch1,selected,selected,selected,selected,selected,selected,selected,selected,selected,selected
AAACAGCCAAGCGATG-1_CCG1112_14_T2_A_CCG1112_MO_Batch4,selected,selected,selected,selected,selected,selected,selected,selected,selected,selected
AAACAGCCAAGCGATG-1_CCG1112_21_T2_A_CCG1112_MO_Batch4,selected,selected,selected,selected,selected,selected,selected,selected,selected,selected
AAACAGCCAATAATGG-1_CCG1112_26_T1_B1_CCG1112_MO_Batch1,selected,selected,selected,selected,selected,selected,selected,selected,unselected,selected


In [7]:
!gsutil cp $RESULT_TABLE/TumorRandom95_10times.csv gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/hr_brca_mo_analysis/data/

Copying file://..//data/table/TumorRandom95_10times.csv [Content-Type=text/csv]...
/ [1 files][  9.0 MiB/  9.0 MiB]                                                
Operation completed over 1 objects/9.0 MiB.                                      


In [1]:
for i in range(10):
    !gsutil cp -r gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/Single_Cell_Cohort/GEX_CCG1112_LowMt/TumorRandom95_$i  /home/analysis/hr_brca_single_cell/data/table/MPs/

Copying gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/Single_Cell_Cohort/GEX_CCG1112_LowMt/TumorRandom95_0/detectrhp.log...
Copying gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/Single_Cell_Cohort/GEX_CCG1112_LowMt/TumorRandom95_0/nmf_basis.pickle...
Copying gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/Single_Cell_Cohort/GEX_CCG1112_LowMt/TumorRandom95_0/nmf_coef.pickle...
Copying gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/Single_Cell_Cohort/GEX_CCG1112_LowMt/TumorRandom95_0/rph_params.json...
| [4 files][ 91.6 MiB/ 91.6 MiB]                                                
Operation completed over 4 objects/91.6 MiB.                                     
Copying gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/Single_Cell_Cohort/GEX_CCG1112_LowMt/TumorRandom95_1/detectrhp.log...
Copying gs://fc-secure-21875d01-8aac-4504-bb02-ae52be667d4a/Single_Cell_Cohort/GEX_CCG1112_LowMt/TumorRandom95_1/nmf_basis.pickle...
Copying gs://fc-secure-21875d01-8aac-4504-bb02-